Generating arrays for h5 file

In [4]:
import glob
import h5py
import importlib
import IPython.display as ipd
import soxr
import numpy as np
import os
import pandas as pd
import pickle
import soundfile as sf
import src.audio_transforms as at
import src.custom_modules as cm
import src.spatial_attn_lightning as binaural_lightning
importlib.reload(binaural_lightning)
import sys
import torch
import tqdm
import yaml

from pathlib import Path
from pytorch_lightning import Trainer
from scipy import signal
from scipy.io.wavfile import read, write

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [5]:
# path to 376 wiki words
swc_path = '/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/foreground_swc/'

# path o common voice words not included in training
# df = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/commonvoice_9_en/manifest_all_word_alignments.pdpkl')

In [6]:
manifest = pd.read_pickle(swc_path + 'manifest.pdpkl')

In [4]:
manifest

,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,raw_clip_dur_in_s,raw_clip_end_in_s,raw_clip_start_in_s,raw_src_fn,raw_total_file_duration_in_s,split,split_int,sr,src_fn,total_file_duration_in_s,word
0,a-c-norman,3.0,3.0,0.0,swc,0.32,2094.94,2094.62,/scratch2/weka/mcdermott/msaddler/swc/english/...,2175.444172,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,above
1,jebjoya,3.0,3.0,0.0,swc,0.49,1715.87,1715.38,/scratch2/weka/mcdermott/msaddler/swc/english/...,2793.356190,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,according
2,caninedoubletake,3.0,3.0,0.0,swc,0.36,169.03,168.67,/scratch2/weka/mcdermott/msaddler/swc/english/...,987.438730,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,across
3,karltalk,3.0,3.0,0.0,swc,0.60,2429.51,2428.91,/scratch2/weka/mcdermott/msaddler/swc/english/...,4802.892336,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,action
4,s-whistler,3.0,3.0,0.0,swc,0.80,1149.87,1149.07,/scratch2/weka/mcdermott/msaddler/swc/english/...,4463.715556,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,activities
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371,incledon,3.0,3.0,0.0,swc,0.69,231.72,231.03,/scratch2/weka/mcdermott/msaddler/swc/english/...,273.573152,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,world
372,ama1016,3.0,3.0,0.0,swc,0.42,193.47,193.05,/scratch2/weka/mcdermott/msaddler/swc/english/...,273.502041,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,writing
373,zanimum,3.0,3.0,0.0,swc,0.27,13.92,13.65,/scratch2/weka/mcdermott/msaddler/swc/english/...,452.154921,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,written
374,tonyle,3.0,3.0,0.0,swc,0.31,1208.98,1208.67,/scratch2/weka/mcdermott/msaddler/swc/english/...,2359.540680,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,wrote


In [22]:
np.random.seed(0)
n_target_words = 360

target_vocab = np.random.choice(manifest.word.unique(), size=n_target_words, replace=False)

In [23]:
fn_pkl_src = '/scratch2/weka/mcdermott/raygon/projects/public/jsinDataset/assets/data/interim/swc/mungedFinalDataframeWords_swc_readerNormalized_sexNormalized_accent.pdpkl'
fn_pkl_dst = '/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl'

In [24]:
word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", 'rb'))
words = list(word_dict.keys())
words = [word.replace("'", "") for word in words]
manifest_all_words = pd.read_pickle(fn_pkl_dst)
# filter out words not in 'words' list
manifest_all_words = manifest_all_words[manifest_all_words['word'].isin(words)]
word_counts = manifest_all_words['word'].value_counts()

(328048, 13)

In [26]:
down_df = manifest_all_words.groupby(['word', 'gender']).sample(1, replace=False)

In [27]:
final_df = down_df.groupby('word').filter(lambda x: len(x) == 2)

In [28]:
final_df = final_df.reset_index().rename(columns={'index':'src_ix'})

In [29]:
len(final_df.word.unique())

789

In [37]:
final_df = final_df[final_df['word'].isin(target_vocab)]

In [38]:
all_words_not_filtered = pd.read_pickle(fn_pkl_dst)
oov_words = all_words_not_filtered[~all_words_not_filtered['word'].isin(words)]

In [39]:
talkers = final_df['client_id'].unique()
oov_words[oov_words['client_id'].isin(talkers)].client_id.value_counts()

s-whistler               60197
matthewdgonzalez         49291
mangst                   36135
the-voice-of-hassocks    35187
alexkillby               26728
                         ...  
enormator                  145
sixteen-left               113
finlay-mcwalter             95
emmiecc                     83
acerperi                    62
Name: client_id, Length: 165, dtype: int64

In [40]:
samples_per_talker = {talker:count for talker,count in final_df.client_id.value_counts().items()}
viables_cues = oov_words[oov_words.client_id.isin(talkers)]

cues = viables_cues.groupby('client_id').apply(lambda group: group.sample(samples_per_talker[group.name]))
cues.drop(columns='client_id', inplace=True)
cues = cues.reset_index()
cues.rename(columns={'level_1':'cue_src_ix'}, inplace=True)

In [41]:
final_df.sort_values(by='client_id', inplace=True)
final_df.reset_index(inplace=True, drop=True)


cues.sort_values(by='client_id', inplace=True)
cues.reset_index(inplace=True, drop=True)



### Merge cues with foregrounds  
cues[['cue_word', 'cue_src_ix', 'cue_client_id', 'cue_fn', 'cue_start_in_s', 'cue_end_in_s']] = cues[['word', 'cue_src_ix', 'client_id', 'src_fn', 'clip_start_in_s', 'clip_end_in_s']]
# Combine as experiment dataframe
tg_and_cue_df = final_df.join(cues[['cue_word', 'cue_src_ix', 'cue_client_id', 'cue_fn', 'cue_start_in_s', 'cue_end_in_s']])
assert (tg_and_cue_df['client_id'] == tg_and_cue_df['cue_client_id']).all(), "Cue and Target talkers don't match!"

In [42]:
tg_and_cue_df

,src_ix,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,gender,gender_int,split,split_int,sr,src_fn,total_file_duration_in_s,word,cue_word,cue_src_ix,cue_client_id,cue_fn,cue_start_in_s,cue_end_in_s
0,992601,1904-cc,0.22,969.29,969.07,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2207.180045,these,dramatic,992746,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,1147.51,1147.96
1,993029,1904-cc,0.30,1989.34,1989.04,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2207.180045,simply,allows,992957,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,1905.81,1906.30
2,1003750,99of9-toby-hudson,0.35,3718.62,3718.27,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,3793.117460,death,with,999423,99of9-toby-hudson,/scratch2/weka/mcdermott/msaddler/swc/english/...,642.92,643.09
3,845840,a-c-norman,0.39,22.72,22.33,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2175.444172,english,purchased,847874,a-c-norman,/scratch2/weka/mcdermott/msaddler/swc/english/...,1953.25,1953.69
4,226810,a-parrot,0.51,1542.51,1542.00,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,3338.366259,system,twelve,226218,a-parrot,/scratch2/weka/mcdermott/msaddler/swc/english/...,905.52,906.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,105077,zanimum,0.55,262.95,262.40,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,617.248798,forces,generated,96674,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,501.23,501.84
716,101856,zanimum,0.40,480.30,479.90,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,3221.953016,times,reference,105698,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,607.63,608.22
717,97804,zanimum,0.44,622.74,622.30,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1220.892154,force,what,98511,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,220.31,220.61
718,105477,zanimum,0.32,300.58,300.26,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,988.727438,chief,actor,99323,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,6.37,6.68


In [43]:
# Get distractor words

distractor_vocab = [word for word in words if word not in target_vocab]

In [94]:
distractor_df = all_words_not_filtered[all_words_not_filtered['word'].isin(distractor_vocab)]
distractor_df


,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,gender,gender_int,split,split_int,sr,src_fn,total_file_duration_in_s,word
1,wodup,0.32,1.49,1.17,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2979.424943,point
43,wodup,0.41,899.78,899.37,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2979.424943,number
44,wodup,0.45,909.58,909.13,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2979.424943,unique
45,wodup,0.34,909.92,909.58,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2979.424943,number
58,wodup,0.26,1035.41,1035.15,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2979.424943,there
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1006249,supa6661,0.49,422.98,422.49,swc,male,1,NaN,0,48000,/scratch2/weka/mcdermott/msaddler/swc/english/...,588.318667,appears
1006251,supa6661,0.73,458.22,457.49,swc,male,1,NaN,0,48000,/scratch2/weka/mcdermott/msaddler/swc/english/...,588.318667,exist
1006259,supa6661,0.31,464.55,464.24,swc,male,1,NaN,0,48000,/scratch2/weka/mcdermott/msaddler/swc/english/...,588.318667,usually
1006271,supa6661,0.40,472.82,472.42,swc,male,1,NaN,0,48000,/scratch2/weka/mcdermott/msaddler/swc/english/...,588.318667,places


array([[     0],
       [     1],
       [     2],
       ...,
       [150404],
       [150405],
       [150406]])

In [109]:
distractors.reset_index()

,gender,level_1,distractor_client_id,distractor_clip_dur_in_s,distractor_clip_end_in_s,distractor_clip_start_in_s,distractor_corpus,distractor_gender,distractor_gender_int,distractor_split,distractor_split_int,distractor_sr,distractor_src_fn,distractor_total_file_duration_in_s,distractor_word
0,female,248234,ama1016,0.30,270.68,270.38,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2128.008707,leave
1,male,488062,runfellow,0.23,1194.83,1194.60,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1681.959184,hundred


In [138]:
bad_ixs = np.argwhere(trial_df.duplicated(subset=['word', 'cue_word','distractor_word']).values)

In [143]:
trial_df.iloc[bad_ixs.flatten()][['word', 'cue_word','distractor_word']]

,word,cue_word,distractor_word
1,class,potential,family
1,according,with,were
1,includes,clearly,were


In [132]:
trial_df[['word', 'cue_word','distractor_word']].eq

,word,cue_word,distractor_word
0,these,dramatic,stars
1,these,dramatic,president
0,simply,allows,offered
1,simply,allows,would
0,death,with,state
...,...,...,...
1,force,what,second
0,chief,actor,origin
1,chief,actor,population
0,attack,into,without


In [163]:
# Get distractors per example 
np.random.seed(0)
target_client_data = tg_and_cue_df[['client_id', 'word']].values
trial_list = []   
for ix, (client_id, word) in enumerate(target_client_data):
    # get subset of distractors that are not the target word or target talker
    distractor_samples = distractor_df[(distractor_df.client_id != client_id) & (distractor_df.word != word) ]
    distractors = distractor_samples.groupby('gender').apply(lambda group: group.sample(1)).reset_index(drop=True)
    # rename distractors column names to include distractor in label
    distractors = distractors.rename(columns={col: 'distractor_' + col for col in distractors.columns})
    # get target row 
    target_row = tg_and_cue_df.iloc[[ix]]
    target_row = target_row.reset_index(drop=True)
    target_row = pd.concat([target_row] * len(distractors), ignore_index=True)
    # merg
    trials = pd.concat([distractors, target_row], axis=1)
    trial_list.append(trials)
    # break

trial_df = pd.concat(trial_list, axis=0, ignore_index=True)

assert trial_df['client_id'].eq(trial_df['cue_client_id']).all(), "Cue and Target talkers don't match!"
assert trial_df['word'].ne(trial_df['cue_word']).all(), "Cue and Target words match!"
assert trial_df['word'].ne(trial_df['distractor_word']).all(), "Distractor and Target words match!"
assert trial_df['client_id'].ne(trial_df['distractor_client_id']).all(), "Distractor and Target talkers match!"


In [180]:
trial_df['gender_cond_td'] = trial_df['gender'] + '_' + trial_df['distractor_gender']

In [182]:
## Check summary stats of trial_df

print(f"N target words {trial_df.word.unique().shape[0]}")
print(f"N trials per target gender\n{trial_df.gender.value_counts()}")
print(f"N trials by target-distractor gender condition \n{trial_df.gender_cond_td.value_counts()}")

N target words 360
N trials per target gender
female    720
male      720
Name: gender, dtype: int64
N trials by target-distractor gender condition 
female_female    360
female_male      360
male_female      360
male_male        360
Name: gender_cond_td, dtype: int64


In [184]:
# specify output path
from pathlib import Path
out_path = Path('/om2/user/imgrisff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/')
out_path.mkdir(exist_ok=True, parents=True)

# save trial manifest
trial_df.to_pickle(out_path / 'full_eval_trial_manifest.pdpkl')

# Cut 2 second excerpts and save stimuli

In [1]:
from pathlib import Path
import pandas as pd 
out_path = Path('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/')
trial_df = pd.read_pickle(out_path / 'full_eval_trial_manifest.pdpkl')

In [2]:
from IPython.display import Audio
from scipy.io.wavfile import read, write
import numpy as np 
import soundfile as sf
import librosa 
import soxr


In [3]:

trial_df['cue_clip_dur_in_s'] = trial_df['cue_end_in_s'] - trial_df['cue_start_in_s']

# rename cue_start_in_s to clip_start_in_s
trial_df.rename(columns={'cue_start_in_s':'cue_clip_start_in_s'}, inplace=True)
# same for end 
trial_df.rename(columns={'cue_end_in_s':'cue_clip_end_in_s'}, inplace=True)
trial_df.rename(columns={'cue_fn':'cue_src_fn'}, inplace=True)



In [4]:
trial_df

,distractor_client_id,distractor_clip_dur_in_s,distractor_clip_end_in_s,distractor_clip_start_in_s,distractor_corpus,distractor_gender,distractor_gender_int,distractor_split,distractor_split_int,distractor_sr,...,total_file_duration_in_s,word,cue_word,cue_src_ix,cue_client_id,cue_src_fn,cue_clip_start_in_s,cue_clip_end_in_s,gender_cond_td,cue_clip_dur_in_s
0,popularoutcast,0.44,359.92,359.48,swc,female,0,NaN,0,44100,...,2207.180045,these,dramatic,992746,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,1147.51,1147.96,female_female,0.45
1,matthewdgonzalez,0.49,584.65,584.16,swc,male,1,NaN,0,44100,...,2207.180045,these,dramatic,992746,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,1147.51,1147.96,female_male,0.45
2,flyingtoaster,0.17,1711.42,1711.25,swc,female,0,NaN,0,44100,...,2207.180045,simply,allows,992957,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,1905.81,1906.30,female_female,0.49
3,warmvoiceover,0.76,487.07,486.31,swc,male,1,NaN,0,44100,...,2207.180045,simply,allows,992957,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,1905.81,1906.30,female_male,0.49
4,popularoutcast,0.56,432.21,431.65,swc,female,0,NaN,0,44100,...,3793.117460,death,with,999423,99of9-toby-hudson,/scratch2/weka/mcdermott/msaddler/swc/english/...,642.92,643.09,male_female,0.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,matthewdgonzalez,0.64,1951.59,1950.95,swc,male,1,NaN,0,44100,...,1220.892154,force,what,98511,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,220.31,220.61,male_male,0.30
1436,popularoutcast,0.62,552.97,552.35,swc,female,0,NaN,0,44100,...,988.727438,chief,actor,99323,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,6.37,6.68,male_female,0.31
1437,incledon,0.49,494.41,493.92,swc,male,1,NaN,0,44100,...,988.727438,chief,actor,99323,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,6.37,6.68,male_male,0.31
1438,popularoutcast,0.62,956.40,955.78,swc,female,0,NaN,0,44100,...,479.647347,attack,into,563727,zegoma-beach,/scratch2/weka/mcdermott/msaddler/swc/english/...,213.10,213.29,male_female,0.19


In [5]:
## Cut and write wav files to directorys and updatae manifest
# specify output path
target_path = Path('/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/target_excerpts')
target_path.mkdir(exist_ok=True, parents=True)

cue_path = Path('/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/cue_excerpts')
cue_path.mkdir(exist_ok=True, parents=True)

distractor_path = Path('/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/distractor_excerpts')
distractor_path.mkdir(exist_ok=True, parents=True)


In [8]:
def pad_or_trim_to_len(x, n, mode='both', kwargs_pad={}):
    """
    Increases or decreases the length of a one-dimensional signal
    by either padding or triming the array. If the difference
    between `len(x)` and `n` is odd, this function will default to
    adding/removing the extra sample at the end of the signal.
    
    Args
    ----
    x (np.ndarray): one-dimensional input signal
    n (int): length of output signal
    mode (str): specify which end of signal to modify
        (default behavior is to symmetrically modify both ends)
    kwargs_pad (dict): keyword arguments for np.pad function
    
    Returns
    -------
    x_out (np.ndarray): one-dimensional signal with length `n`
    """
    assert len(np.array(x).shape) == 1, "input must be 1D array"
    assert mode.lower() in ['both', 'start', 'end']
    n_diff = np.abs(len(x) - n)
    if len(x) > n:
        if mode.lower() == 'end':
            x_out = x[:n]
        elif mode.lower() == 'start':
            x_out = x[-n:]
        else:
            x_out = x[int(np.floor(n_diff / 2)):-int(np.ceil(n_diff / 2))]
    elif len(x) < n:
        if mode.lower() == 'end':
            pad_width = [0, n_diff]
        elif mode.lower() == 'start':
            pad_width = [n_diff, 0]
        else:
            pad_width = [int(np.floor(n_diff / 2)), int(np.ceil(n_diff / 2))]
        kwargs = {'mode': 'constant'}
        kwargs.update(kwargs_pad)
        x_out = np.pad(x, pad_width, **kwargs)
    else:
        x_out = x
    assert len(x_out) == n
    return x_out
 

def get_excerpt(dfi, dur=3.0, sr=50000, pad_with_context=True, jitter_fraction=0):
    """
    This function loads an audio file and excerpts a clip with the specified
    duration. Target durations that exceed clip boundaries are handled with
    zero-padding (applied to all signals but sliced away when not needed).
    This function also handles resampling (via soxr) and re-scaling.
    """
    jitter_in_s = 0
    jitter_via_zero_padding = True
    if dfi.clip_dur_in_s > dur:
        # Take a random segment if clip duration is longer than excerpt
        clip_start_in_s = np.random.uniform(
            low=dfi.clip_start_in_s,
            high=dfi.clip_start_in_s + dfi.clip_dur_in_s - dur,
            size=None)
        clip_end_in_s = clip_start_in_s + dur
        jitter_via_zero_padding = False
    else:
        # Temporally jitter clip by extending either start or end time
        jitter_in_s = np.random.uniform(
            low=-dfi.clip_dur_in_s * jitter_fraction,
            high=dfi.clip_dur_in_s * jitter_fraction,
            size=None)
        if pad_with_context:
            # If using context, adjust clip start and end times to account for jitter and context
            if jitter_in_s > 0:
                clip_start_in_s = dfi.clip_start_in_s - (2 * np.abs(jitter_in_s))
                clip_end_in_s = dfi.clip_end_in_s
            else:
                clip_start_in_s = dfi.clip_start_in_s
                clip_end_in_s = dfi.clip_end_in_s + (2 * np.abs(jitter_in_s))
            clip_dur_in_s = clip_end_in_s - clip_start_in_s
            jitter_via_zero_padding = False
            context_pad_in_s = (dur - clip_dur_in_s) / 2
        else:
            clip_start_in_s = dfi.clip_start_in_s
            clip_end_in_s = dfi.clip_end_in_s
            context_pad_in_s = 0
        clip_start_in_s = clip_start_in_s - context_pad_in_s
        clip_end_in_s = clip_end_in_s + context_pad_in_s
    clip_dur_in_s = clip_end_in_s - clip_start_in_s
    load_full_file = True
    total_file_duration_in_s = librosa.get_duration(filename=dfi.src_fn)
    if (clip_start_in_s >= 0) and (clip_end_in_s < total_file_duration_in_s):
        # Attempt to read only the specified excerpt
        myfile = sf.SoundFile(dfi.src_fn)
        if myfile.seekable():
            src_sr = myfile.samplerate
            frame_start = int(np.round(clip_start_in_s * src_sr))
            frames = int(np.round(clip_dur_in_s * src_sr))
            myfile.seek(frame_start)
            y = myfile.read(frames, always_2d=True)
            y = np.mean(y, axis=1)
            load_full_file = False
    if load_full_file:
        # If impossible, read full audio file
        y, src_sr = sf.read(dfi.src_fn, always_2d=True)
        y = np.mean(y, axis=1)
        frame_start = int(np.round(clip_start_in_s * src_sr))
        frames = int(np.round(clip_dur_in_s * src_sr))
        if frame_start < 0:
            y = np.pad(y, [-frame_start, 0])
            frame_start = 0
        if frame_start + frames > len(y):
            y = np.pad(y, [0, frame_start + frames - len(y)])
        y = y[frame_start : frame_start + frames]
    # Resample from src_sr to sr
    y = soxr.resample(y, src_sr, sr).astype(np.float32)
    # If not yet jittered, apply jitter at end via asymmetric zero-padding
    if jitter_via_zero_padding:
        jitter_pad_width = int(np.round(2 * np.abs(jitter_in_s) * sr))
        if jitter_in_s > 0:
            y = np.pad(y, [jitter_pad_width, 0]).astype(np.float32)
        else:
            y = np.pad(y, [0, jitter_pad_width]).astype(np.float32)
    # Zero-pad or trim to length (fixes off by one errors)
    y = pad_or_trim_to_len(y, int(dur * sr))
    y = np.nan_to_num(y.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return y

In [9]:
## Use excerpt function to cut and write wav files to directorys and updatae manifest
from tqdm.auto import tqdm
# make distractor_df which is df of columns of traial_df with distractor_ prefix in column name

distractor_df = trial_df.filter(regex='distractor_')
distractor_df.columns = distractor_df.columns.str.replace('distractor_', '')

# do the same for cue_df
cue_df = trial_df.filter(regex='cue_')
cue_df.columns = cue_df.columns.str.replace('cue_', '')

sr = 44100
duration = 2.0
for ix in tqdm(range(len(trial_df)), total=len(trial_df)):
    # get row from trial_df
    dfi = trial_df.iloc[ix]
    # get excerpt of target
    target_excerpt = get_excerpt(dfi, dur=duration, jitter_fraction=0)
    # get excerpt of cue
    cue_excerpt = get_excerpt(cue_df.iloc[ix], dur=duration, jitter_fraction=0)
    # get excerpt of distractor
    distractor_excerpt = get_excerpt(distractor_df.iloc[ix], dur=duration, jitter_fraction=0)
    # write target excerpt to file
    target_fn = target_path / f"{dfi.word}_{dfi.client_id}.wav"
    write(target_fn, sr, target_excerpt)
    # write cue excerpt to file
    cue_fn = cue_path / f"{dfi.cue_word}_{dfi.cue_client_id}.wav"
    write(cue_fn, sr, cue_excerpt)
    # write distractor excerpt to file
    distractor_fn = distractor_path / f"{dfi.distractor_word}_{dfi.distractor_client_id}.wav"
    write(distractor_fn, sr, distractor_excerpt)
    # update manifest with new file names
    trial_df.loc[ix, 'src_fn'] = str(target_fn)
    trial_df.loc[ix, 'cue_src_fn'] = str(cue_fn)
    trial_df.loc[ix, 'distractor_src_fn'] = str(distractor_fn)

trial_df.to_pickle(out_path / 'full_eval_trial_manifest_new_fnames.pdpkl')


  0%|          | 0/1440 [00:00<?, ?it/s]

In [10]:
trial_df.head()

,distractor_client_id,distractor_clip_dur_in_s,distractor_clip_end_in_s,distractor_clip_start_in_s,distractor_corpus,distractor_gender,distractor_gender_int,distractor_split,distractor_split_int,distractor_sr,...,total_file_duration_in_s,word,cue_word,cue_src_ix,cue_client_id,cue_src_fn,cue_clip_start_in_s,cue_clip_end_in_s,gender_cond_td,cue_clip_dur_in_s
0,popularoutcast,0.44,359.92,359.48,swc,female,0,NaN,0,44100,...,2207.180045,these,dramatic,992746,1904-cc,/om/user/imgriff/datasets/spatial_audio_pipeli...,1147.51,1147.96,female_female,0.45
1,matthewdgonzalez,0.49,584.65,584.16,swc,male,1,NaN,0,44100,...,2207.180045,these,dramatic,992746,1904-cc,/om/user/imgriff/datasets/spatial_audio_pipeli...,1147.51,1147.96,female_male,0.45
2,flyingtoaster,0.17,1711.42,1711.25,swc,female,0,NaN,0,44100,...,2207.180045,simply,allows,992957,1904-cc,/om/user/imgriff/datasets/spatial_audio_pipeli...,1905.81,1906.30,female_female,0.49
3,warmvoiceover,0.76,487.07,486.31,swc,male,1,NaN,0,44100,...,2207.180045,simply,allows,992957,1904-cc,/om/user/imgriff/datasets/spatial_audio_pipeli...,1905.81,1906.30,female_male,0.49
4,popularoutcast,0.56,432.21,431.65,swc,female,0,NaN,0,44100,...,3793.117460,death,with,999423,99of9-toby-hudson,/om/user/imgriff/datasets/spatial_audio_pipeli...,642.92,643.09,male_female,0.17


In [14]:
# Make sure audio worked 
eg_row = trial_df.iloc[[20]]
print(eg_row.word)
Audio(eg_row['src_fn'].values[0])

20    music
Name: word, dtype: object


In [15]:
print(eg_row.cue_word)
Audio(eg_row['cue_src_fn'].values[0])

20    movements
Name: cue_word, dtype: object


In [16]:
print(eg_row.distractor_word)
Audio(eg_row['distractor_src_fn'].values[0])

20    could
Name: distractor_word, dtype: object


In [19]:
out_path = Path('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/')
trial_df = pd.read_pickle(out_path / 'full_eval_trial_manifest_new_fnames.pdpkl')

In [20]:
(out_path / 'full_eval_trial_manifest_new_fnames.pdpkl').as_posix()

'/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/full_eval_trial_manifest_new_fnames.pdpkl'

In [21]:
trial_df

,distractor_client_id,distractor_clip_dur_in_s,distractor_clip_end_in_s,distractor_clip_start_in_s,distractor_corpus,distractor_gender,distractor_gender_int,distractor_split,distractor_split_int,distractor_sr,...,total_file_duration_in_s,word,cue_word,cue_src_ix,cue_client_id,cue_src_fn,cue_clip_start_in_s,cue_clip_end_in_s,gender_cond_td,cue_clip_dur_in_s
0,popularoutcast,0.44,359.92,359.48,swc,female,0,NaN,0,44100,...,2207.180045,these,dramatic,992746,1904-cc,/om/user/imgriff/datasets/spatial_audio_pipeli...,1147.51,1147.96,female_female,0.45
1,matthewdgonzalez,0.49,584.65,584.16,swc,male,1,NaN,0,44100,...,2207.180045,these,dramatic,992746,1904-cc,/om/user/imgriff/datasets/spatial_audio_pipeli...,1147.51,1147.96,female_male,0.45
2,flyingtoaster,0.17,1711.42,1711.25,swc,female,0,NaN,0,44100,...,2207.180045,simply,allows,992957,1904-cc,/om/user/imgriff/datasets/spatial_audio_pipeli...,1905.81,1906.30,female_female,0.49
3,warmvoiceover,0.76,487.07,486.31,swc,male,1,NaN,0,44100,...,2207.180045,simply,allows,992957,1904-cc,/om/user/imgriff/datasets/spatial_audio_pipeli...,1905.81,1906.30,female_male,0.49
4,popularoutcast,0.56,432.21,431.65,swc,female,0,NaN,0,44100,...,3793.117460,death,with,999423,99of9-toby-hudson,/om/user/imgriff/datasets/spatial_audio_pipeli...,642.92,643.09,male_female,0.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,matthewdgonzalez,0.64,1951.59,1950.95,swc,male,1,NaN,0,44100,...,1220.892154,force,what,98511,zanimum,/om/user/imgriff/datasets/spatial_audio_pipeli...,220.31,220.61,male_male,0.30
1436,popularoutcast,0.62,552.97,552.35,swc,female,0,NaN,0,44100,...,988.727438,chief,actor,99323,zanimum,/om/user/imgriff/datasets/spatial_audio_pipeli...,6.37,6.68,male_female,0.31
1437,incledon,0.49,494.41,493.92,swc,male,1,NaN,0,44100,...,988.727438,chief,actor,99323,zanimum,/om/user/imgriff/datasets/spatial_audio_pipeli...,6.37,6.68,male_male,0.31
1438,popularoutcast,0.62,956.40,955.78,swc,female,0,NaN,0,44100,...,479.647347,attack,into,563727,zegoma-beach,/om/user/imgriff/datasets/spatial_audio_pipeli...,213.10,213.29,male_female,0.19
